# Phase 1 Backtest Results

Loads `output/backtest/cohorts/{cohort_id}/results.jsonl` and
computes Information Coefficient (Spearman ρ), Hit Rate, and
Decile Sort across the 4 forward horizons (1M / 3M / 6M / 12M).

**Prerequisites:**

- `pip install pandas matplotlib jupyter`
- Cohort run completed (state.json + results.jsonl present).

**Defaults:** `cohort_id = "2025Q1"`. Override the constant in
the next cell to analyze a different cohort.


In [ ]:
from __future__ import annotations
import json
import pathlib
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
while ROOT.name and not (ROOT / "tools" / "backtest").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("Run this notebook from inside the stock-analysis-agent repo")
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tools.backtest.metrics import (
    compute_decile_sort,
    compute_hit_rate,
    compute_ic,
)
from tools.paths import data_dir

COHORT_ID = "2025Q1"  # change to "smoke" for the 1-ticker test cohort
HORIZONS = ("1m", "3m", "6m", "12m")


In [ ]:
results_path = data_dir() / "backtest" / "cohorts" / COHORT_ID / "results.jsonl"
if not results_path.exists():
    raise FileNotFoundError(
        f"results.jsonl not found at {results_path}. "
        f"Did you run `python tools/backtest_runner.py --cohort {COHORT_ID}` "
        f"and then the cohort aggregator?"
    )

rows = [json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines() if line.strip()]
df = pd.DataFrame(rows)
print(f"Loaded {len(df)} rows from {results_path}")
print(f"Columns: {list(df.columns)}")
df.head()


## Information Coefficient (Spearman ρ)

Spearman rank correlation between `rr_score` and excess returns
at each horizon. Higher = the analyst's R/R Score is a stronger
predictor of forward excess returns. Approximate two-sided
p-value (normal approximation; rough for n < 30).


In [ ]:
ic_rows = []
for h in HORIZONS:
    col = f"excess_{h}"
    if col not in df.columns:
        continue
    res = compute_ic(df, score_col="rr_score", return_col=col)
    ic_rows.append({
        "horizon": h,
        "n": res.n,
        "spearman_rho": res.spearman_rho,
        "p_value": res.p_value,
    })

ic_df = pd.DataFrame(ic_rows).set_index("horizon")
ic_df


## Hit Rate

Directional accuracy of bullish (`비중확대` / `Buy`) and bearish
(`비중축소` / `Sell`) verdicts vs realized 12-month returns.
Neutral verdicts (`중립` / `Hold`) are excluded. Zero-return
ties count as misses.


In [ ]:
hr_rows = []
for h in HORIZONS:
    col = f"return_{h}"
    if col not in df.columns:
        continue
    res = compute_hit_rate(df, verdict_col="verdict", return_col=col)
    hr_rows.append({
        "horizon": h,
        "n_total": res.n_total,
        "hit_rate": res.hit_rate,
        "bullish_n": res.bullish_n,
        "bullish_hit_rate": res.bullish_hit_rate,
        "bearish_n": res.bearish_n,
        "bearish_hit_rate": res.bearish_hit_rate,
        "neutral_excluded": res.n_neutral_excluded,
        "nan_excluded": res.n_nan_excluded,
    })

hr_df = pd.DataFrame(hr_rows).set_index("horizon")
hr_df


## Decile Sort

Buckets tickers into 10 deciles by `rr_score` and reports mean
`excess_12m` per decile. A monotonic increase from bucket 1
(lowest scores) to bucket 10 (highest scores) is the bull case
for the system. The "top minus bottom" spread is the headline
number.


In [ ]:
ds = compute_decile_sort(df, score_col="rr_score", return_col="excess_12m", n_buckets=10)
print(f"n_buckets actual: {ds.n_buckets}, n dropped (NaN): {ds.n_dropped_nan}")
print(f"Top minus bottom spread: {ds.top_minus_bottom_spread}")

ds_df = pd.DataFrame(ds.buckets)
ds_df


In [ ]:
if ds.n_buckets >= 2:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.scatter(ds_df["bucket"], ds_df["mean_return"] * 100, s=80, color="steelblue")
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_xlabel("Decile (1 = lowest rr_score, 10 = highest)")
    ax.set_ylabel("Mean excess_12m return (%)")
    ax.set_title(f"{COHORT_ID}: Mean excess_12m return by R/R Score decile")
    ax.set_xticks(range(1, ds.n_buckets + 1))
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(f"Not enough buckets to plot (n_buckets={ds.n_buckets}). Cohort too small or scores too clustered.")


## Verdict-level mean returns

Mean realized 12-month return per verdict label. A useful
sanity check: bullish verdicts should have higher mean returns
than neutral, which should be higher than bearish.


In [ ]:
if "verdict" in df.columns and "return_12m" in df.columns:
    verdict_means = (
        df.dropna(subset=["return_12m"])
        .groupby("verdict")["return_12m"]
        .agg(["mean", "count"])
        .sort_values("mean")
    )
    print(verdict_means)
    if len(verdict_means) > 0:
        fig, ax = plt.subplots(figsize=(9, 4.5))
        colors = ["crimson" if x < 0 else "seagreen" for x in verdict_means["mean"]]
        ax.bar(verdict_means.index.astype(str), verdict_means["mean"] * 100, color=colors)
        ax.axhline(0, color="grey", linewidth=0.5)
        ax.set_xlabel("Verdict")
        ax.set_ylabel("Mean return_12m (%)")
        ax.set_title(f"{COHORT_ID}: Mean 12M return by verdict")
        for i, (label, row) in enumerate(verdict_means.iterrows()):
            ax.text(i, row["mean"] * 100, f" n={int(row['count'])}", ha="center", va="bottom", fontsize=8)
        plt.tight_layout()
        plt.show()
else:
    print("verdict or return_12m column missing — skipping bar chart")


## Summary

After Phase 1 cohort runs, fill this section by hand:

- Headline IC (12M, Spearman ρ): _________
- Headline Hit Rate (12M): _________
- Decile spread (top − bottom): _________
- Top 3 surprises / failure cases: _________

See `evals/backtest/reports/2026-05-08-phase1-report.md` (Chunk 6
Task 6.3) for the formal write-up.
